In [1]:
# import ML_CV_Teaching.lane_detection.lane_pkg as lp
import matplotlib.pyplot as plt
import cv2
import numpy as np
%matplotlib inline

In [2]:
import numpy as np

def region_of_interest(img, vertices):
    mask = np.zeros_like(img)
    if len(img.shape) > 2:
        channel_count = img.shape[2]
        ignore_mask_color = (255,) * channel_count
    else:
        ignore_mask_color = 255

    cv2.fillPoly(mask, vertices, ignore_mask_color)
    masked_image = cv2.bitwise_and(img, mask)
    return masked_image

def draw_lines(img, lines, color=(0, 255, 0), thickness=2):
    line_img = np.zeros_like(img)
    if lines is None:
        return None
    for line in lines:
        for x1, y1, x2, y2 in line:
            cv2.line(line_img, (x1, y1), (x2, y2), color, thickness)
    return line_img

def process_image(image):
    imshape = image.shape

    # 1. Grayscale
    grayscaleImage = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # 2. Gaussian Blur
    gaussianImage = cv2.GaussianBlur(grayscaleImage, (5, 5), 0)

    # 3. Canny Edge Detection
    cannyImage = cv2.Canny(gaussianImage, 50, 150)

    # 4. Region of Interest Masking
    roiMask = np.array([[
        (0, imshape[0]),
        (465, 320),
        (475, 320),
        (imshape[1], imshape[0])
    ]], dtype=np.int32)
    edgeMask = region_of_interest(cannyImage, roiMask)

    # 5. Hough Transform for line detection
    rho = 2
    theta = np.pi / 180
    threshold = 15
    min_line_len = 40
    max_line_gap = 20

    raw_lines = cv2.HoughLinesP(
        edgeMask, rho, theta, threshold,
        np.array([]), minLineLength=min_line_len,
        maxLineGap=max_line_gap
    )

    # 6. Store lines
    stored_lines = []
    if raw_lines is not None:
        for line in raw_lines:
            x1, y1, x2, y2 = line[0]
            stored_lines.append(((x1, y1), (x2, y2)))

    # 7. Draw lines
    linesImage = draw_lines(image, raw_lines)

    if linesImage is None:
        return image, []

    # 8. Overlay detected lines on original image
    result = cv2.addWeighted(image, 0.8, linesImage, 1, 0)
    return result, stored_lines


# Direct link with openCV

In [3]:
# if you want to display processed_img inline in Jupyter, without any crashing errors use:
import matplotlib.pyplot as plt


import cv2
import numpy as np
import time

# open ZED2 whole camera
cap = cv2.VideoCapture('/dev/video0')

if not cap.isOpened():
    print("Error: Could not open video stream.")
    exit()

while True:
    ret, frame = cap.read()
    if not ret:
        print("Failed to grab frame.")
        break
    
    # resize the frame for left half processing (optional)
    height, width = frame.shape[:2]
    left_half = frame[:, :width // 2]

    try:
        processed_img, lines = process_image(left_half)
    except Exception as e:
        print(f"Processing error: {e}")
        processed_img = left_half
        lines = []

    # if you want better visualization, uncomment the following line:
    cv2.imshow("Processed ZED2 Camera", processed_img)

    # if you want to display processed_img inline in Jupyter, without any crashing errors use:
    # plt.imshow(cv2.cvtColor(processed_img, cv2.COLOR_BGR2RGB))
    # plt.axis('off')
    # plt.show()

    print(f"Lines detected this frame: {len(lines)}")
    print(f"Useful data points on these frames: {lines}")

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
    
    # the lesser the sleep time, the more the chances of crashing
    time.sleep(0.15)

cap.release()
cv2.destroyAllWindows()


Lines detected this frame: 0
Useful data points on these frames: []
Lines detected this frame: 0
Useful data points on these frames: []
Lines detected this frame: 0
Useful data points on these frames: []
Lines detected this frame: 0
Useful data points on these frames: []
Lines detected this frame: 0
Useful data points on these frames: []
Lines detected this frame: 0
Useful data points on these frames: []
Lines detected this frame: 0
Useful data points on these frames: []
Lines detected this frame: 0
Useful data points on these frames: []
Lines detected this frame: 0
Useful data points on these frames: []
Lines detected this frame: 0
Useful data points on these frames: []
Lines detected this frame: 0
Useful data points on these frames: []
Lines detected this frame: 0
Useful data points on these frames: []
Lines detected this frame: 0
Useful data points on these frames: []
Lines detected this frame: 0
Useful data points on these frames: []
Lines detected this frame: 0
Useful data points 

# Triangularisation Method
# Avoid this if not necessary

In [7]:
# use this for triangulation when using lane detection 
!git clone https://github.com/wwang/ML_CV_Teaching.git
# on your local machine, go to lane_pkg.py and remove the movie.py import

Cloning into 'ML_CV_Teaching'...
remote: Enumerating objects: 168, done.
remote: Counting objects: 100% (58/58), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 168 (delta 20), reused 47 (delta 9), pack-reused 110 (from 1)
Receiving objects: 100% (168/168), 81.87 MiB | 18.95 MiB/s, done.
Resolving deltas: 100% (65/65), done.


In [10]:
import ML_CV_Teaching.lane_detection.lane_pkg as lp
import matplotlib.pyplot as plt
import cv2
import numpy as np
%matplotlib inline

In [11]:
def process_image(image):
    imshape = image.shape
    grayscaleImage = lp.grayscale(image)
    gaussianImage = lp.gaussian_blur(grayscaleImage, 5)
    cannyImage = lp.canny(gaussianImage, 50, 150)
    roiMask = np.array([[(0, imshape[0]), (465, 320), (475, 320), (imshape[1], imshape[0])]], dtype=np.int32)
    edgeMask = lp.region_of_interest(cannyImage, roiMask)

    rho = 2
    theta = np.pi/180
    threshold = 15
    min_line_len = 40
    max_line_gap = 20

    linesImage = lp.hough_lines(edgeMask, rho, theta, threshold, min_line_len, max_line_gap)

    # get raw lines for storage
    raw_lines = cv2.HoughLinesP(edgeMask, rho, theta, threshold, np.array([]), minLineLength=min_line_len, maxLineGap=max_line_gap)

    stored_lines = []
    if raw_lines is not None:
        for line in raw_lines:
            x1, y1, x2, y2 = line[0]
            stored_lines.append(((x1, y1), (x2, y2)))

    if linesImage is None:
        return image, []

    result = cv2.addWeighted(image, 0.8, linesImage, 1, 0)
    return result, stored_lines

In [ ]:
import matplotlib.pyplot as plt


import cv2
import numpy as np
import time

# open ZED2 whole camera
cap = cv2.VideoCapture('/dev/video0')

if not cap.isOpened():
    print("Error: Could not open video stream.")
    exit()

while True:
    ret, frame = cap.read()
    if not ret:
        print("Failed to grab frame.")
        break
    
    # resize the frame for left half processing (optional)
    height, width = frame.shape[:2]
    left_half = frame[:, :width // 2]

    try:
        processed_img, lines = process_image(left_half)
    except Exception as e:
        print(f"Processing error: {e}")
        processed_img = left_half
        lines = []

    # if you want better visualization, uncomment the following line:
    cv2.imshow("Processed ZED2 Camera", processed_img)

    # if you want to display processed_img inline in Jupyter, without any crashing errors use:
    # plt.imshow(cv2.cvtColor(processed_img, cv2.COLOR_BGR2RGB))
    # plt.axis('off')
    # plt.show()

    print(f"Lines detected this frame: {len(lines)}")
    print(f"Useful data points on these frames: {lines}")

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
    
    # the lesser the sleep time, the more the chances of crashing
    time.sleep(0.15)

cap.release()
cv2.destroyAllWindows()

Processing error: 'NoneType' object is not iterable
Lines detected this frame: 0
Useful data points on these frames: []
Processing error: 'NoneType' object is not iterable
Lines detected this frame: 0
Useful data points on these frames: []
Processing error: 'NoneType' object is not iterable
Lines detected this frame: 0
Useful data points on these frames: []
Processing error: 'NoneType' object is not iterable
Lines detected this frame: 0
Useful data points on these frames: []
Processing error: 'NoneType' object is not iterable
Lines detected this frame: 0
Useful data points on these frames: []
Processing error: 'NoneType' object is not iterable
Lines detected this frame: 0
Useful data points on these frames: []
Processing error: 'NoneType' object is not iterable
Lines detected this frame: 0
Useful data points on these frames: []
Processing error: 'NoneType' object is not iterable
Lines detected this frame: 0
Useful data points on these frames: []
Processing error: 'NoneType' object is n

/home/arrk-adas/Desktop/Rc-Car/src/stage_ros2/src/ML_CV_Teaching/lane_detection/lane_pkg.py:92: RuntimeWarning: divide by zero encountered in scalar divide
  slope = (y1 - y2) / (x1 - x2)
/home/arrk-adas/Desktop/Rc-Car/src/stage_ros2/src/ML_CV_Teaching/lane_detection/lane_pkg.py:107: RuntimeWarning: divide by zero encountered in scalar divide
  slope = (y1 - y2)/(x1 - x2)
/home/arrk-adas/Desktop/Rc-Car/src/stage_ros2/src/ML_CV_Teaching/lane_detection/lane_pkg.py:115: RuntimeWarning: divide by zero encountered in scalar divide
  slope = (y1 - y2)/(x1 - x2)
/home/arrk-adas/miniforge3/envs/lane_env/lib/python3.10/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/arrk-adas/miniforge3/envs/lane_env/lib/python3.10/site-packages/numpy/_core/_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/arrk-adas/miniforge3/envs/lane_env/lib/python3

Lines detected this frame: 1
Useful data points on these frames: [((np.int32(671), np.int32(375)), (np.int32(671), np.int32(333)))]
Lines detected this frame: 1
Useful data points on these frames: [((np.int32(671), np.int32(375)), (np.int32(671), np.int32(335)))]
Processing error: 'NoneType' object is not iterable
Lines detected this frame: 0
Useful data points on these frames: []
Lines detected this frame: 1
Useful data points on these frames: [((np.int32(671), np.int32(375)), (np.int32(671), np.int32(333)))]
Processing error: 'NoneType' object is not iterable
Lines detected this frame: 0
Useful data points on these frames: []
Lines detected this frame: 4
Useful data points on these frames: [((np.int32(228), np.int32(372)), (np.int32(341), np.int32(365))), ((np.int32(439), np.int32(359)), (np.int32(652), np.int32(333))), ((np.int32(997), np.int32(371)), (np.int32(1046), np.int32(357))), ((np.int32(307), np.int32(368)), (np.int32(386), np.int32(362)))]
Processing error: 'NoneType' obje

/home/arrk-adas/miniforge3/envs/lane_env/lib/python3.10/site-packages/numpy/_core/_methods.py:191: RuntimeWarning: invalid value encountered in subtract
  x = asanyarray(arr - arrmean)
/home/arrk-adas/Desktop/Rc-Car/src/stage_ros2/src/ML_CV_Teaching/lane_detection/lane_pkg.py:72: RuntimeWarning: invalid value encountered in scalar subtract
  if slope - meanSlope < 1.5 * sdSlope:


Lines detected this frame: 2
Useful data points on these frames: [((np.int32(972), np.int32(354)), (np.int32(1017), np.int32(374))), ((np.int32(671), np.int32(375)), (np.int32(671), np.int32(333)))]
Lines detected this frame: 4
Useful data points on these frames: [((np.int32(882), np.int32(346)), (np.int32(937), np.int32(374))), ((np.int32(265), np.int32(374)), (np.int32(306), np.int32(339))), ((np.int32(898), np.int32(347)), (np.int32(951), np.int32(373))), ((np.int32(900), np.int32(347)), (np.int32(954), np.int32(374)))]
Lines detected this frame: 6
Useful data points on these frames: [((np.int32(506), np.int32(330)), (np.int32(643), np.int32(372))), ((np.int32(505), np.int32(334)), (np.int32(622), np.int32(372))), ((np.int32(862), np.int32(345)), (np.int32(918), np.int32(374))), ((np.int32(877), np.int32(346)), (np.int32(935), np.int32(375))), ((np.int32(253), np.int32(375)), (np.int32(293), np.int32(343))), ((np.int32(589), np.int32(361)), (np.int32(630), np.int32(374)))]
Lines det

/home/arrk-adas/Desktop/Rc-Car/src/stage_ros2/src/ML_CV_Teaching/lane_detection/lane_pkg.py:124: RuntimeWarning: invalid value encountered in scalar divide
  intersection = (negIntercept - posIntercept) / (posCoef - negCoef)


Lines detected this frame: 1
Useful data points on these frames: [((np.int32(452), np.int32(325)), (np.int32(513), np.int32(322)))]
Lines detected this frame: 4
Useful data points on these frames: [((np.int32(402), np.int32(333)), (np.int32(479), np.int32(331))), ((np.int32(672), np.int32(334)), (np.int32(777), np.int32(339))), ((np.int32(461), np.int32(331)), (np.int32(507), np.int32(329))), ((np.int32(363), np.int32(334)), (np.int32(404), np.int32(334)))]
Lines detected this frame: 5
Useful data points on these frames: [((np.int32(749), np.int32(341)), (np.int32(824), np.int32(342))), ((np.int32(320), np.int32(337)), (np.int32(398), np.int32(337))), ((np.int32(459), np.int32(334)), (np.int32(573), np.int32(326))), ((np.int32(672), np.int32(335)), (np.int32(716), np.int32(339))), ((np.int32(672), np.int32(375)), (np.int32(672), np.int32(333)))]
Lines detected this frame: 4
Useful data points on these frames: [((np.int32(740), np.int32(343)), (np.int32(831), np.int32(344))), ((np.int32